# P1 — Judge runs (Colab / Kaggle)
Run top to bottom. **Runtime → Change runtime type → T4 GPU.**
Every experiment cell is safe to re-run: finished calls are skipped, so after a disconnect just reconnect, run the setup cells (1–4), and re-run the experiment cell.

## 1. Settings

In [ ]:
PLATFORM = "colab"          # "colab" or "kaggle"
BACKEND  = "vllm"           # "vllm" first; switch to "hf" if vLLM won't install/run within ~1 hour
REPO_URL = "https://github.com/<your-org>/peogai-judge.git"   # <-- P3's repo

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/peogai-judge"
    OUT  = "/content/drive/MyDrive/peogai/results/raw"     # checkpoints live on Drive
else:  # Kaggle: /kaggle/working survives as notebook output; download it after each session
    WORK = "/kaggle/working/peogai-judge"
    OUT  = "/kaggle/working/results_raw"
import os; os.makedirs(OUT, exist_ok=True)

## 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Get the code + data

In [ ]:
import os
if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
%cd {WORK}
!ls data prompts

## 4. Install
**vLLM:** installing it may replace Colab's torch; if you see an import error afterwards, do *Runtime → Restart session* and run from cell 1 again (skip this cell).
**HF fallback:** set `BACKEND = "hf"` in cell 1.

In [ ]:
if BACKEND == "vllm":
    !pip install -q vllm
else:
    !pip install -q -U transformers accelerate bitsandbytes
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|bitsandbytes) "

## 4b. Build the data (P1 does this now; P2 reviews it later)
Samples 150 CodeJudgeBench code-generation pairs (seed 42, balanced by difficulty) and writes `data/pairs_150.jsonl` + `data/pairs_150_stats.json` (numbers for the data slide).
Then builds `data/pairs_150_verbose.jsonl` for Exp 2. **Commit both files** so everyone uses the exact same sample.

In [ ]:
import os
if not os.path.exists("data/pairs_150.jsonl"):
    !pip install -q datasets
    !python -m src.sample_data
!python -m src.perturb
!cat data/pairs_150_stats.json

In [ ]:
# Long answers get cut at 8k tokens. Check how many pairs come close (~4 chars per token).
import json
rows = [json.loads(l) for l in open("data/pairs_150.jsonl")]
lens = sorted(len(r["problem"]) + len(r["correct"]) + len(r["incorrect"]) for r in rows)
print("max chars:", lens[-1], "| pairs over ~28k chars:", sum(l > 28000 for l in lens))

## 5. Smoke test — 5 pairs, both models (note the s/call)

In [ ]:
RUN = f"python -m src.run_experiment --backend {BACKEND} --data data/pairs_150.jsonl --out {OUT}"
for m in ["3b", "7b"]:
    !{RUN} --exp exp1 --model {m} --limit 5 --tag _smoke

In [ ]:
# Eyeball raw outputs: are the verdicts parsing? Any rambling?
import json
for m in ["3b", "7b"]:
    rows = [json.loads(l) for l in open(f"{OUT}/exp1_{m}_smoke.jsonl")]
    print(m, [(r["job_id"], repr(r["raw"][:40]), r["verdict"]) for r in rows][:10])

## 6. Full runs
Rough budget (to be confirmed by the smoke test): 1,800 judge calls in total; exp4c generates 3 samples per call.
Each `!python` call is a fresh process, so GPU memory is freed between models.

### Exp 1 — pairwise, both orders (300 calls / model)

In [ ]:
!{RUN} --exp exp1 --model 3b
!{RUN} --exp exp1 --model 7b

### Exp 3 — pointwise, all 300 answers

In [ ]:
!{RUN} --exp exp3 --model 3b
!{RUN} --exp exp3 --model 7b

### Exp 4b — debias prompt, both orders

In [ ]:
!{RUN} --exp exp4b --model 3b
!{RUN} --exp exp4b --model 7b

### Exp 4c — majority of 3 @ T=0.7, both orders (3B)

In [ ]:
!{RUN} --exp exp4c --model 3b

### Exp 2 — verbosity (3B, if time). Uses `data/pairs_150_verbose.jsonl` from cell 4b.

In [ ]:
!{RUN} --exp exp2 --model 3b --data data/pairs_150_verbose.jsonl

## 7. Numbers for the Experimental Setup section

In [ ]:
!python -m src.setup_summary --raw {OUT}

## 8. Hand results to P2 (copy into the repo and push)

In [ ]:
!mkdir -p results/raw && cp {OUT}/*.jsonl {OUT}/*_meta.json results/raw/
# !git add results/raw && git commit -m "P1: raw judge results" && git push